# YETI JSON to CSV
Converts every JSON file in `data/raw_yeti_response/` into a flattened CSV in `data/csv/`. Drop new JSON files into the raw folder and run all cells.

In [ ]:
import json
from pathlib import Path
import pandas as pd


PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "data/raw_yeti_response").is_dir()
)
RAW_DIR = PROJECT_ROOT / "data/raw_yeti_response"
CSV_DIR = PROJECT_ROOT / "data/csv"
CSV_DIR.mkdir(parents=True, exist_ok=True)

# Set to True to regenerate CSVs that already exist.
OVERWRITE = False

In [ ]:
def json_to_dataframe(path):
    pages = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(pages, dict):
        pages = [pages]
    rows = [row for page in pages for row in page["data"]["data"]]
    df = pd.json_normalize(rows)
    for col in df.columns:
        if df[col].map(lambda x: isinstance(x, list)).any():
            df[col] = df[col].map(lambda x: json.dumps(x) if isinstance(x, list) else x)
    return df

for json_path in sorted(RAW_DIR.glob("*.json")):
    out_path = CSV_DIR / f"{json_path.stem}.csv"
    if out_path.exists() and not OVERWRITE:
        print(f"skipped (exists): {out_path.name}")
        continue
    df = json_to_dataframe(json_path)
    df.to_csv(out_path, index=False)
    print(f"wrote {out_path.name}: {df.shape[0]} rows x {df.shape[1]} columns")